<a href="https://colab.research.google.com/github/krishRajawat/My-Google-Colab-Work/blob/main/Anomaly_Detection_using_CS_ELM_by_Krish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =======================================================================
# "High-Speed Anomaly Detection using CS-ELM"
# Author: Krish Rajawat
# Focus: Manual Implementation of Class-Specific Extreme Learning Machine
# =======================================================================

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
import time

# 1. GENERATE IMBALANCED ANOMALY DATA
# 1000 samples, 20 features, 5% Anomaly (Class 1)
X, y = make_classification(n_samples=2000, n_features=20, n_clusters_per_class=1,
                           weights=[0.95], flip_y=0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. THE CS-ELM CLASS (Manual NumPy Implementation)
class CSELM:
    def __init__(self, n_hidden_nodes, weight_c):
        self.L = n_hidden_nodes
        self.C = weight_c # Penalty parameter for minority class

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        self.N, self.n_features = X.shape
        # A. Randomly assign input weights and biases
        self.W = np.random.normal(size=(self.n_features, self.L))
        self.b = np.random.normal(size=(self.L))

        # B. Calculate Hidden Layer Output Matrix (H)
        H = self.sigmoid(np.dot(X, self.W) + self.b)

        # C. Create Class-Specific Weight Matrix (W_mat)
        # We give higher weight to the minority class (1)
        w_diag = np.ones(self.N)
        minority_indices = np.where(y == 1)[0]
        w_diag[minority_indices] = self.C
        W_mat = np.diag(w_diag)

        # D. Solve for Output Weights (beta) using Moore-Penrose Pseudoinverse
        # Formula: beta = (I/C + H'.W.H)^-1 . H'.W.T
        T = (y * 2 - 1).reshape(-1, 1) # Convert to [-1, 1] for ELM

        part1 = np.eye(self.L) + np.dot(H.T, np.dot(W_mat, H))
        part2 = np.dot(H.T, np.dot(W_mat, T))
        self.beta = np.linalg.solve(part1, part2)

    def predict(self, X):
        H = self.sigmoid(np.dot(X, self.W) + self.b)
        out = np.dot(H, self.beta)
        return (out > 0).astype(int).flatten()

# 3. TRAINING & BENCHMARKING
start_time = time.time()
# L=100 hidden nodes, Penalty C=50 for minority class
model = CSELM(n_hidden_nodes=100, weight_c=50)
model.fit(X_train, y_train)
end_time = time.time()

# 4. RESULTS
y_pred = model.predict(X_test)
print(f"CS-ELM Training Time: {end_time - start_time:.4f} seconds")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print("\nDetailed Performance:")
print(classification_report(y_test, y_pred))

CS-ELM Training Time: 0.1913 seconds
F1-Score: 0.3415

Detailed Performance:
              precision    recall  f1-score   support

           0       1.00      0.79      0.88       378
           1       0.21      0.95      0.34        22

    accuracy                           0.80       400
   macro avg       0.60      0.87      0.61       400
weighted avg       0.95      0.80      0.85       400



Note: I created this personal Project while studying my Imbalance learning course!😁

Most people use Deep Learning which relies on Gradient Descent (Iterative). I chose an Extreme Learning Machine (ELM) because it randomly assigns input weights and only solves for output weights analytically. For imbalanced data, I modified it into a Class-Specific ELM. By adding a diagonal weight matrix to the optimization problem, I could mathematically force the network to 'care' more about the minority class without needing multiple training epochs. It’s significantly faster and mathematically more elegant for real-time anomaly detection.